## 1. Sorting & Ranking 
Sorting is essential before grouping, visualising, or reporting. Pandas provides fast, flexible sort methods. 
## 1.1 sort_values & sort_index

In [2]:
import seaborn as sns
df = sns.load_dataset('titanic')

print("Passengers sorted by age (ascending):")
print(df.sort_values('age').head())

print("Passengers sorted by fare (descending):")
print(df.sort_values('fare', ascending=False).head())

print("Sorted by pclass ascending and age descending:")
print(
    df.sort_values(
        ['pclass', 'age'],
        ascending=[True, False]
    ).head(10)
)

df_shuffled = df.sample(frac=1, random_state=42)

print("Sorted by index:")
print(df_shuffled.sort_index().head())

print("Top 5 highest fares:")
print(df.nlargest(5, 'fare')[['who', 'fare', 'pclass']])

print("Top 5 youngest passengers:")
print(df.nsmallest(5, 'age')[['who', 'age', 'pclass']])

Passengers sorted by age (ascending):
     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
803         1       3    male  0.42      0      1   8.5167        C   Third   
755         1       2    male  0.67      1      1  14.5000        S  Second   
644         1       3  female  0.75      2      1  19.2583        C   Third   
469         1       3  female  0.75      2      1  19.2583        C   Third   
78          1       2    male  0.83      0      2  29.0000        S  Second   

       who  adult_male deck  embark_town alive  alone  
803  child       False  NaN    Cherbourg   yes  False  
755  child       False  NaN  Southampton   yes  False  
644  child       False  NaN    Cherbourg   yes  False  
469  child       False  NaN    Cherbourg   yes  False  
78   child       False  NaN  Southampton   yes  False  
Passengers sorted by fare (descending):
     survived  pclass     sex   age  sibsp  parch      fare embarked  class  \
679         1       1    male  3

## 1.2 Ranking 

In [3]:
df = sns.load_dataset('titanic').copy()

df['fare_rank'] = df['fare'].rank(
    ascending=False,
    method='dense'
)

print("Fare ranking:")
print(
    df[['who', 'fare', 'fare_rank']]
    .sort_values('fare_rank')
    .head()
)

df['fare_pct'] = df['fare'].rank(pct=True)

print("Percentile rank of fares:")
print(df[['fare', 'fare_pct']].head())

Fare ranking:
       who      fare  fare_rank
679    man  512.3292        1.0
737    man  512.3292        1.0
258  woman  512.3292        1.0
341  woman  263.0000        2.0
88   woman  263.0000        2.0
Percentile rank of fares:
      fare  fare_pct
0   7.2500  0.086420
1  71.2833  0.885522
2   7.9250  0.260943
3  53.1000  0.839506
4   8.0500  0.296296


## 1.3 Tasks — Sorting & Ranking 
Task 1: Sort Titanic by fare descending. Who are the top 10 highest-paying passengers? Print their name (if available), fare, and class. 

Task 2: Sort by pclass ascending, then by age descending within each class. Show the youngest in each class. 

Task 3: Add a rank column to a 10-student marks DataFrame (rank 1=best). Try all 5 rank methods and compare results for tied students.

In [4]:
print("Task 1")

df = sns.load_dataset('titanic')

top10 = df.nlargest(10, 'fare')[['alive', 'fare', 'pclass']]

print("Top 10 highest-paying passengers:")
print(top10)

Task 1
Top 10 highest-paying passengers:
    alive      fare  pclass
258   yes  512.3292       1
679   yes  512.3292       1
737   yes  512.3292       1
27     no  263.0000       1
88    yes  263.0000       1
341   yes  263.0000       1
438    no  263.0000       1
311   yes  262.3750       1
742   yes  262.3750       1
118    no  247.5208       1


In [5]:
print("Task 2")

df = sns.load_dataset('titanic')

sorted_df = df.sort_values(
    ['pclass', 'age'],
    ascending=[True, False]
)

youngest = sorted_df.loc[
    sorted_df.groupby('pclass')['age'].idxmin()
]

print("Youngest passenger in each class:")
print(youngest[['pclass', 'age', 'who']])

Task 2
Youngest passenger in each class:
     pclass   age    who
305       1  0.92  child
755       2  0.67  child
803       3  0.42  child


In [7]:
import pandas as pd
print("Task 3")

students = pd.DataFrame({
    'Name': [
        'A', 'B', 'C', 'D', 'E',
        'F', 'G', 'H', 'I', 'J'
    ],
    'Marks': [90, 85, 90, 78, 85, 95, 72, 95, 85, 78]
})

students['Rank_Average'] = students['Marks'].rank(
    ascending=False,
    method='average'
)

students['Rank_Min'] = students['Marks'].rank(
    ascending=False,
    method='min'
)

students['Rank_Max'] = students['Marks'].rank(
    ascending=False,
    method='max'
)

students['Rank_Dense'] = students['Marks'].rank(
    ascending=False,
    method='dense'
)

students['Rank_First'] = students['Marks'].rank(
    ascending=False,
    method='first'
)

print("Student ranking using different methods:")
print(students)

Task 3
Student ranking using different methods:
  Name  Marks  Rank_Average  Rank_Min  Rank_Max  Rank_Dense  Rank_First
0    A     90           3.5       3.0       4.0         2.0         3.0
1    B     85           6.0       5.0       7.0         3.0         5.0
2    C     90           3.5       3.0       4.0         2.0         4.0
3    D     78           8.5       8.0       9.0         4.0         8.0
4    E     85           6.0       5.0       7.0         3.0         6.0
5    F     95           1.5       1.0       2.0         1.0         1.0
6    G     72          10.0      10.0      10.0         5.0        10.0
7    H     95           1.5       1.0       2.0         1.0         2.0
8    I     85           6.0       5.0       7.0         3.0         7.0
9    J     78           8.5       8.0       9.0         4.0         9.0


## 2. GroupBy — Split · Apply · Combine 
GroupBy is one of Pandas' most powerful features. It follows the split-apply-combine pattern used in SQL, Excel pivot tables, and R's dplyr. 
## 2.1 Basic Aggregation

In [8]:
df = sns.load_dataset('titanic')

print("Average fare by passenger class:")
print(df.groupby('pclass')['fare'].mean())

print("Total survivors by sex:")
print(df.groupby('sex')['survived'].sum())

print("Age statistics by passenger class:")
print(
    df.groupby('pclass')['age']
    .agg(['mean', 'std', 'count', 'min', 'max'])
)

print("Survival rate by class and sex:")
print(
    df.groupby(['pclass', 'sex'])['survived']
    .mean()
    .unstack()
)

result = df.groupby('pclass').agg(
    avg_age=('age', 'mean'),
    total_fare=('fare', 'sum'),
    survival_pct=('survived', 'mean'),
    passengers=('survived', 'count')
)

print("Grouped summary:")
print(result.round(2))

Average fare by passenger class:
pclass
1    84.154687
2    20.662183
3    13.675550
Name: fare, dtype: float64
Total survivors by sex:
sex
female    233
male      109
Name: survived, dtype: int64
Age statistics by passenger class:
             mean        std  count   min   max
pclass                                         
1       38.233441  14.802856    186  0.92  80.0
2       29.877630  14.001077    173  0.67  70.0
3       25.140620  12.495398    355  0.42  74.0
Survival rate by class and sex:
sex       female      male
pclass                    
1       0.968085  0.368852
2       0.921053  0.157407
3       0.500000  0.135447
Grouped summary:
        avg_age  total_fare  survival_pct  passengers
pclass                                               
1         38.23    18177.41          0.63         216
2         29.88     3801.84          0.47         184
3         25.14     6714.70          0.24         491


## 2.2 Transform & Filter 

In [9]:
df = sns.load_dataset('titanic').copy()

df['class_mean_fare'] = df.groupby('pclass')['fare'].transform('mean')

df['fare_vs_class'] = (
    df['fare'] - df['class_mean_fare']
)

df['fare_z'] = df.groupby('pclass')['fare'].transform(
    lambda x: (x - x.mean()) / x.std()
)

big_groups = df.groupby('pclass').filter(
    lambda x: len(x) > 100
)

print("Passenger count in large groups:")
print(big_groups['pclass'].value_counts())

def top_earners(group, n=3):
    return group.nlargest(n, 'fare')

top = df.groupby('pclass').apply(top_earners)

print("Top earners by class:")
print(top[['pclass', 'who', 'fare']])

Passenger count in large groups:
pclass
3    491
1    216
2    184
Name: count, dtype: int64
Top earners by class:
            pclass    who      fare
pclass                             
1      258       1  woman  512.3292
       679       1    man  512.3292
       737       1    man  512.3292
2      72        2    man   73.5000
       120       2    man   73.5000
       385       2    man   73.5000
3      159       3    man   69.5500
       180       3  woman   69.5500
       201       3    man   69.5500


C:\Users\prince patel\AppData\Local\Temp\ipykernel_25684\667649356.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top = df.groupby('pclass').apply(top_earners)


## 2.3 Tasks — GroupBy 
Task 1: On Titanic: compute survival rate by sex, by pclass, and by (sex × pclass) combination. Which group had the highest survival rate? 

Task 2: For each passenger class, find the mean, median, and std of fares. Use a single .agg() call. 
  
Task 3: Add a column class_median_age = median age within each pclass using transform. Then add a flag IsOlderThanClassMedian. 
  
Task 4: Filter out passenger classes with fewer than 200 passengers. How many passengers remain? 

Task 5: Group by embarked port and compute: total passengers, survival rate, average fare. Sort by survival rate descending. 

In [10]:
print("Task 1")

df = sns.load_dataset('titanic')

print("Survival rate by sex:")
print(df.groupby('sex')['survived'].mean())

print("Survival rate by passenger class:")
print(df.groupby('pclass')['survived'].mean())

survival = df.groupby(
    ['sex', 'pclass']
)['survived'].mean()

print("Survival rate by sex and class:")
print(survival)

print("Group with highest survival rate:")
print(survival.idxmax())

Task 1
Survival rate by sex:
sex
female    0.742038
male      0.188908
Name: survived, dtype: float64
Survival rate by passenger class:
pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
Survival rate by sex and class:
sex     pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: survived, dtype: float64
Group with highest survival rate:
('female', np.int64(1))


In [11]:
print("Task 2")

df = sns.load_dataset('titanic')

fare_stats = df.groupby('pclass')['fare'].agg(
    ['mean', 'median', 'std']
)

print("Fare statistics by passenger class:")
print(fare_stats)

Task 2
Fare statistics by passenger class:
             mean   median        std
pclass                               
1       84.154687  60.2875  78.380373
2       20.662183  14.2500  13.417399
3       13.675550   8.0500  11.778142


In [12]:
print("Task 3")

df = sns.load_dataset('titanic')

df['class_median_age'] = (
    df.groupby('pclass')['age']
    .transform('median')
)

df['IsOlderThanClassMedian'] = (
    df['age'] > df['class_median_age']
)

print(df[
    ['age', 'pclass',
     'class_median_age',
     'IsOlderThanClassMedian']
].head())

Task 3
    age  pclass  class_median_age  IsOlderThanClassMedian
0  22.0       3              24.0                   False
1  38.0       1              37.0                    True
2  26.0       3              24.0                    True
3  35.0       1              37.0                   False
4  35.0       3              24.0                    True


In [13]:
print("Task 4")

df = sns.load_dataset('titanic')

filtered = df.groupby('pclass').filter(
    lambda x: len(x) >= 200
)

print("Remaining passengers:")
print(len(filtered))

Task 4
Remaining passengers:
707


In [14]:
print("Task 5")

df = sns.load_dataset('titanic')

result = df.groupby('embarked').agg(
    total_passengers=('survived', 'count'),
    survival_rate=('survived', 'mean'),
    average_fare=('fare', 'mean')
)

result = result.sort_values(
    'survival_rate',
    ascending=False
)

print("Grouped statistics by embarked port:")
print(result)

Task 5
Grouped statistics by embarked port:
          total_passengers  survival_rate  average_fare
embarked                                               
C                      168       0.553571     59.954144
Q                       77       0.389610     13.276030
S                      644       0.336957     27.079812


## 3. Merging & Joining DataFrames 
## 3.1 pd.merge — SQL-style Joins

In [15]:
students = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Carol', 'Dave'],
    'dept': ['CS', 'Math', 'CS', 'Phy']
})

scores = pd.DataFrame({
    'id': [1, 2, 3, 5],
    'score': [88, 75, 92, 65]
})

inner = pd.merge(students, scores, on='id', how='inner')

left = pd.merge(students, scores, on='id', how='left')

right = pd.merge(students, scores, on='id', how='right')

outer = pd.merge(students, scores, on='id', how='outer')

print("Left join:")
print(left)

scores2 = scores.rename(columns={'id': 'student_id'})

merged = pd.merge(
    students,
    scores2,
    left_on='id',
    right_on='student_id'
)

print("Merge with different column names:")
print(merged)

Left join:
   id   name  dept  score
0   1  Alice    CS   88.0
1   2    Bob  Math   75.0
2   3  Carol    CS   92.0
3   4   Dave   Phy    NaN
Merge with different column names:
   id   name  dept  student_id  score
0   1  Alice    CS           1     88
1   2    Bob  Math           2     75
2   3  Carol    CS           3     92


## 3.2 pd.concat

In [16]:
df1 = pd.DataFrame({
    'A': [1, 2],
    'B': ['x', 'y']
})

df2 = pd.DataFrame({
    'A': [3, 4],
    'B': ['z', 'w']
})

df3 = pd.DataFrame({
    'C': [10, 20],
    'D': [30, 40]
})

vertical = pd.concat(
    [df1, df2],
    ignore_index=True
)

print("Vertical concatenation:")
print(vertical)

horizontal = pd.concat(
    [df1, df3],
    axis=1
)

print("Horizontal concatenation:")
print(horizontal)

labeled = pd.concat(
    [df1, df2],
    keys=['batch1', 'batch2']
)

print("Concatenation with keys:")
print(labeled)

print("Rows belonging to batch1:")
print(labeled.loc['batch1'])

Vertical concatenation:
   A  B
0  1  x
1  2  y
2  3  z
3  4  w
Horizontal concatenation:
   A  B   C   D
0  1  x  10  30
1  2  y  20  40
Concatenation with keys:
          A  B
batch1 0  1  x
       1  2  y
batch2 0  3  z
       1  4  w
Rows belonging to batch1:
   A  B
0  1  x
1  2  y


## 3.3 Tasks — Merge & Concat

Task 1: Create a Students table (id, name, age) and a Grades table (id, subject, marks). Perform all 4 join types and explain the row count difference.

Task 2: You have two monthly sales DataFrames (same columns). Concatenate them into a single yearly DataFrame. Add a Month column before concatenating. 

Task 3: Merge Titanic with a separate DataFrame mapping pclass → class_label ('First','Second','Third'). Verify all rows have a label. 

Task 4: Detect duplicate rows after a merge. Use df.duplicated() to find and df.drop_duplicates() to remove them.

In [17]:
print("Task 1")

students = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Carol', 'Dave'],
    'age': [20, 21, 22, 23]
})

grades = pd.DataFrame({
    'id': [1, 2, 3, 5],
    'subject': ['Math', 'Science', 'English', 'Physics'],
    'marks': [85, 90, 88, 75]
})

print("Inner Join:")
print(pd.merge(students, grades, on='id', how='inner'))

print("Left Join:")
print(pd.merge(students, grades, on='id', how='left'))

print("Right Join:")
print(pd.merge(students, grades, on='id', how='right'))

print("Outer Join:")
print(pd.merge(students, grades, on='id', how='outer'))

Task 1
Inner Join:
   id   name  age  subject  marks
0   1  Alice   20     Math     85
1   2    Bob   21  Science     90
2   3  Carol   22  English     88
Left Join:
   id   name  age  subject  marks
0   1  Alice   20     Math   85.0
1   2    Bob   21  Science   90.0
2   3  Carol   22  English   88.0
3   4   Dave   23      NaN    NaN
Right Join:
   id   name   age  subject  marks
0   1  Alice  20.0     Math     85
1   2    Bob  21.0  Science     90
2   3  Carol  22.0  English     88
3   5    NaN   NaN  Physics     75
Outer Join:
   id   name   age  subject  marks
0   1  Alice  20.0     Math   85.0
1   2    Bob  21.0  Science   90.0
2   3  Carol  22.0  English   88.0
3   4   Dave  23.0      NaN    NaN
4   5    NaN   NaN  Physics   75.0


In [18]:
print("Task 2")

jan_sales = pd.DataFrame({
    'Product': ['A', 'B', 'C'],
    'Sales': [1000, 1200, 900]
})

feb_sales = pd.DataFrame({
    'Product': ['A', 'B', 'C'],
    'Sales': [1100, 1300, 950]
})

jan_sales['Month'] = 'January'
feb_sales['Month'] = 'February'

yearly_sales = pd.concat(
    [jan_sales, feb_sales],
    ignore_index=True
)

print("Yearly sales DataFrame:")
print(yearly_sales)

Task 2
Yearly sales DataFrame:
  Product  Sales     Month
0       A   1000   January
1       B   1200   January
2       C    900   January
3       A   1100  February
4       B   1300  February
5       C    950  February


In [19]:
print("Task 3")

df = sns.load_dataset('titanic')

class_labels = pd.DataFrame({
    'pclass': [1, 2, 3],
    'class_label': ['First', 'Second', 'Third']
})

merged_df = pd.merge(
    df,
    class_labels,
    on='pclass',
    how='left'
)

print("First few rows after merge:")
print(merged_df[['pclass', 'class_label']].head())

print("Missing labels:")
print(merged_df['class_label'].isnull().sum())

Task 3
First few rows after merge:
   pclass class_label
0       3       Third
1       1       First
2       3       Third
3       1       First
4       3       Third
Missing labels:
0


In [20]:
print("Task 4")

df1 = pd.DataFrame({
    'id': [1, 2, 2, 3],
    'name': ['A', 'B', 'B', 'C']
})

print("Original DataFrame:")
print(df1)

duplicates = df1[df1.duplicated()]

print("Duplicate rows:")
print(duplicates)

df1 = df1.drop_duplicates()

print("After removing duplicates:")
print(df1)

Task 4
Original DataFrame:
   id name
0   1    A
1   2    B
2   2    B
3   3    C
Duplicate rows:
   id name
2   2    B
After removing duplicates:
   id name
0   1    A
1   2    B
3   3    C


## 4. Pivot Tables & Crosstab 
## 4.1 pivot_table 

In [21]:
df = sns.load_dataset('titanic')

pt = pd.pivot_table(
    df,
    values='fare',
    index='pclass',
    columns='sex',
    aggfunc='mean',
    margins=True
)

print("Average fare by class and sex:")
print(pt.round(2))

pt2 = pd.pivot_table(
    df,
    values='survived',
    index='pclass',
    columns='sex',
    aggfunc=['mean', 'count']
)

print("Survival statistics:")
print(pt2)

pt3 = pd.pivot_table(
    df,
    values=['fare', 'age', 'survived'],
    index='pclass',
    aggfunc='mean'
)

print("Average fare, age and survival rate by class:")
print(pt3.round(2))

Average fare by class and sex:
sex     female   male    All
pclass                      
1       106.13  67.23  84.15
2        21.97  19.74  20.66
3        16.12  12.66  13.68
All      44.48  25.52  32.20
Survival statistics:
            mean            count     
sex       female      male female male
pclass                                
1       0.968085  0.368852     94  122
2       0.921053  0.157407     76  108
3       0.500000  0.135447    144  347
Average fare, age and survival rate by class:
          age   fare  survived
pclass                        
1       38.23  84.15      0.63
2       29.88  20.66      0.47
3       25.14  13.68      0.24


## 4.2 crosstab 

In [22]:
df = sns.load_dataset('titanic')

ct = pd.crosstab(df['pclass'], df['survived'])

print("Frequency table:")
print(ct)

ct_pct = pd.crosstab(
    df['pclass'],
    df['survived'],
    normalize='index'
) * 100

print("Percentage survival by class:")
print(ct_pct.round(1))

ct_m = pd.crosstab(
    df['sex'],
    df['survived'],
    margins=True
)

print("Crosstab with margins:")
print(ct_m)

ct3 = pd.crosstab(
    [df['sex'], df['pclass']],
    df['survived']
)

print("Three-way crosstab:")
print(ct3)

Frequency table:
survived    0    1
pclass            
1          80  136
2          97   87
3         372  119
Percentage survival by class:
survived     0     1
pclass              
1         37.0  63.0
2         52.7  47.3
3         75.8  24.2
Crosstab with margins:
survived    0    1  All
sex                    
female     81  233  314
male      468  109  577
All       549  342  891
Three-way crosstab:
survived         0   1
sex    pclass         
female 1         3  91
       2         6  70
       3        72  72
male   1        77  45
       2        91  17
       3       300  47


## 4.3 Tasks — Pivot & Crosstab 

Task 1: Build a pivot table: mean and std of age grouped by pclass (rows) and survived (columns). Include totals. 

Task 2: Use crosstab to show percentage survival by sex AND embarked port. Which combination had the highest survival rate? 

Task 3: Pivot Titanic to show total fare collected per class per embarkation port. Which port generated the most revenue in each class?

In [23]:
print("Task 1")

df = sns.load_dataset('titanic')

pt = pd.pivot_table(
    df,
    values='age',
    index='pclass',
    columns='survived',
    aggfunc=['mean', 'std'],
    margins=True
)

print("Mean and standard deviation of age:")
print(pt.round(2))

Task 1
Mean and standard deviation of age:
           mean                  std              
survived      0      1    All      0      1    All
pclass                                            
1         43.70  35.37  38.23  15.28  13.76  14.80
2         33.54  25.90  29.88  12.15  14.84  14.00
3         26.56  20.65  25.14  12.33  12.00  12.50
All       30.63  28.34  29.70  14.17  14.95  14.53


In [24]:
print("Task 2")

df = sns.load_dataset('titanic')

ct = pd.crosstab(
    [df['sex'], df['embarked']],
    df['survived'],
    normalize='index'
) * 100

print("Percentage survival by sex and embarked port:")
print(ct.round(2))

highest = ct[1].idxmax()

print("Combination with highest survival rate:")
print(highest)

Task 2
Percentage survival by sex and embarked port:
survived             0      1
sex    embarked              
female C         12.33  87.67
       Q         25.00  75.00
       S         31.03  68.97
male   C         69.47  30.53
       Q         92.68   7.32
       S         82.54  17.46
Combination with highest survival rate:
('female', 'C')


In [25]:
print("Task 3")

df = sns.load_dataset('titanic')

pt = pd.pivot_table(
    df,
    values='fare',
    index='pclass',
    columns='embarked',
    aggfunc='sum'
)

print("Total fare collected by class and embarkation port:")
print(pt.round(2))

print("Port generating maximum revenue in each class:")
print(pt.idxmax(axis=1))

Task 3
Total fare collected by class and embarkation port:
embarked        C       Q        S
pclass                            
1         8901.08  180.00  8936.34
2          431.09   37.05  3333.70
3          740.13  805.20  5169.36
Port generating maximum revenue in each class:
pclass
1    S
2    S
3    S
dtype: object


## 5. Apply, Map & Custom Transformations 
## 5.1 map, apply, applymap

In [27]:
import numpy as np
df = sns.load_dataset('titanic').copy()

df['sex_code'] = df['sex'].map({
    'male': 0,
    'female': 1
})

df['age_sq'] = df['age'].apply(
    lambda x: x**2 if pd.notna(x) else np.nan
)

def classify(row):

    if row['survived'] == 1 and row['age'] < 18:
        return 'Saved Child'

    elif row['survived'] == 1:
        return 'Saved Adult'

    else:
        return 'Not Saved'


df['status'] = df.apply(classify, axis=1)

print("Status counts:")
print(df['status'].value_counts())

nums = df.select_dtypes('number')

rounded = nums.applymap(
    lambda x: round(x, 1) if pd.notna(x) else x
)

print("Rounded numeric values:")
print(rounded.head())

df['fare_band'] = pd.cut(
    df['fare'],
    bins=[0, 10, 50, 200, 600],
    labels=['Budget', 'Economy', 'Premium', 'Luxury']
)

print("Fare bands:")
print(df[['fare', 'fare_band']].head())

Status counts:
status
Not Saved      549
Saved Adult    281
Saved Child     61
Name: count, dtype: int64
Rounded numeric values:
   survived  pclass   age  sibsp  parch  fare  sex_code  age_sq
0         0       3  22.0      1      0   7.2         0   484.0
1         1       1  38.0      1      0  71.3         1  1444.0
2         1       3  26.0      0      0   7.9         1   676.0
3         1       1  35.0      1      0  53.1         1  1225.0
4         0       3  35.0      0      0   8.1         0  1225.0
Fare bands:
      fare fare_band
0   7.2500    Budget
1  71.2833   Premium
2   7.9250    Budget
3  53.1000   Premium
4   8.0500    Budget


C:\Users\prince patel\AppData\Local\Temp\ipykernel_25684\3769642011.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  rounded = nums.applymap(


## 5.2 Tasks — Apply & Map 

Task 1: Use map to replace Titanic 'embarked' codes (C/Q/S) with full names (Cherbourg/Queenstown/Southampton). 

Task 2: Use apply (axis=1) to create a column 'Outcome': 'Survived-Rich' (survived & fare>50), 'Survived Poor', 'Died-Rich', 'Died-Poor'. 

Task 3: Use pd.cut to bin ages into 5 equal-width bands. Then use pd.qcut for 5 quantile-based bands. Compare the distribution. 

Task 4: Without using loops, use applymap (or df.map in newer pandas) to round every numeric cell in a DataFrame to 2 decimal places. 

In [28]:
print("Task 1")

df = sns.load_dataset('titanic')

df['embarked_full'] = df['embarked'].map({
    'C': 'Cherbourg',
    'Q': 'Queenstown',
    'S': 'Southampton'
})

print("Embarked codes with full names:")
print(df[['embarked', 'embarked_full']].head())

Task 1
Embarked codes with full names:
  embarked embarked_full
0        S   Southampton
1        C     Cherbourg
2        S   Southampton
3        S   Southampton
4        S   Southampton


In [29]:
print("Task 2")

df = sns.load_dataset('titanic')

def outcome(row):

    if row['survived'] == 1 and row['fare'] > 50:
        return 'Survived-Rich'

    elif row['survived'] == 1:
        return 'Survived-Poor'

    elif row['fare'] > 50:
        return 'Died-Rich'

    else:
        return 'Died-Poor'


df['Outcome'] = df.apply(outcome, axis=1)

print("Outcome counts:")
print(df['Outcome'].value_counts())

Task 2
Outcome counts:
Outcome
Died-Poor        498
Survived-Poor    233
Survived-Rich    109
Died-Rich         51
Name: count, dtype: int64


In [30]:
print("Task 3")

df = sns.load_dataset('titanic')

df['Age_Band_Cut'] = pd.cut(
    df['age'],
    bins=5
)

df['Age_Band_QCut'] = pd.qcut(
    df['age'],
    q=5
)

print("Distribution using pd.cut():")
print(df['Age_Band_Cut'].value_counts())

print("Distribution using pd.qcut():")
print(df['Age_Band_QCut'].value_counts())

Task 3
Distribution using pd.cut():
Age_Band_Cut
(16.336, 32.252]    346
(32.252, 48.168]    188
(0.34, 16.336]      100
(48.168, 64.084]     69
(64.084, 80.0]       11
Name: count, dtype: int64
Distribution using pd.qcut():
Age_Band_QCut
(0.419, 19.0]    164
(31.8, 41.0]     144
(41.0, 80.0]     142
(19.0, 25.0]     137
(25.0, 31.8]     127
Name: count, dtype: int64


In [31]:
print("Task 4")

df = pd.DataFrame({
    'A': [1.23456, 2.98765],
    'B': [3.45678, 4.76543]
})

print("Original DataFrame:")
print(df)

rounded_df = df.applymap(
    lambda x: round(x, 2)
)

print("Rounded DataFrame:")
print(rounded_df)

Task 4
Original DataFrame:
         A        B
0  1.23456  3.45678
1  2.98765  4.76543
Rounded DataFrame:
      A     B
0  1.23  3.46
1  2.99  4.77


C:\Users\prince patel\AppData\Local\Temp\ipykernel_25684\977934747.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  rounded_df = df.applymap(


## 6. String Operations — str Accessor

In [32]:
df = pd.DataFrame({
    'Name': ['  Alice Smith', 'BOB Jones ', 'carol LEE'],
    'Email': ['alice@mail.com', 'bob@work.org', 'carol@mail.com'],
    'City': ['New York-NY', 'Los Angeles-CA', 'Chicago-IL']
})

df['Name'] = df['Name'].str.strip().str.title()

df['Email'] = df['Email'].str.lower()

df[['City_Name', 'State']] = df['City'].str.split(
    '-',
    expand=True
)

mail_users = df[df['Email'].str.contains('@mail')]

print("Users with @mail addresses:")
print(mail_users)

df['name_len'] = df['Name'].str.len()

df['City'] = df['City'].str.replace('-', ' — ')

df['Surname'] = df['Name'].str.extract(r'(\w+)$')

print("Updated DataFrame:")
print(df)

Users with @mail addresses:
          Name           Email         City City_Name State
0  Alice Smith  alice@mail.com  New York-NY  New York    NY
2    Carol Lee  carol@mail.com   Chicago-IL   Chicago    IL
Updated DataFrame:
          Name           Email              City    City_Name State  name_len  \
0  Alice Smith  alice@mail.com     New York — NY     New York    NY        11   
1    Bob Jones    bob@work.org  Los Angeles — CA  Los Angeles    CA         9   
2    Carol Lee  carol@mail.com      Chicago — IL      Chicago    IL         9   

  Surname  
0   Smith  
1   Jones  
2     Lee  


## 6.1 Tasks — String Operations 

Task 1: Load a CSV with messy names (mixed case, extra spaces). Clean: strip, title-case, extract first and last name into separate columns. 

Task 2: From a product code column like 'AB-1234-XY', extract: category (first 2 chars), ID (middle 4 digits), suffix (last 2 chars). 

Task 3: From Titanic 'who' column, use str.contains to find all passengers whose name contains 'Mr' or 'Mrs'. Count each group. 

Task 4: From an email column, extract the domain (e.g., 'gmail' from 'user@gmail.com') using str.extract with a regex group. 

In [33]:
print("Task 1")

df = pd.DataFrame({
    'Name': [
        '  alice smith',
        'BOB JONES ',
        ' carol lee '
    ]
})

df['Name'] = df['Name'].str.strip().str.title()

df[['First_Name', 'Last_Name']] = df['Name'].str.split(
    ' ',
    expand=True
)

print("Cleaned DataFrame:")
print(df)

Task 1
Cleaned DataFrame:
          Name First_Name Last_Name
0  Alice Smith      Alice     Smith
1    Bob Jones        Bob     Jones
2    Carol Lee      Carol       Lee


In [34]:
print("Task 2")

df = pd.DataFrame({
    'Product_Code': [
        'AB-1234-XY',
        'CD-5678-ZW',
        'EF-9012-MN'
    ]
})

df['Category'] = df['Product_Code'].str[:2]

df['ID'] = df['Product_Code'].str.extract(r'(\d{4})')

df['Suffix'] = df['Product_Code'].str[-2:]

print("Extracted components:")
print(df)

Task 2
Extracted components:
  Product_Code Category    ID Suffix
0   AB-1234-XY       AB  1234     XY
1   CD-5678-ZW       CD  5678     ZW
2   EF-9012-MN       EF  9012     MN


In [35]:
print("Task 3")

df = pd.DataFrame({
    'Name': [
        'Mr John Smith',
        'Mrs Alice Brown',
        'Miss Emma Lee',
        'Mr David Clark',
        'Mrs Sarah White'
    ]
})

mr_count = df['Name'].str.contains('Mr ').sum()

mrs_count = df['Name'].str.contains('Mrs ').sum()

print("Number of Mr:")
print(mr_count)

print("Number of Mrs:")
print(mrs_count)

Task 3
Number of Mr:
2
Number of Mrs:
2


In [36]:
print("Task 4")

df = pd.DataFrame({
    'Email': [
        'user1@gmail.com',
        'user2@yahoo.com',
        'user3@outlook.com'
    ]
})

df['Domain'] = df['Email'].str.extract(
    r'@([^.]+)'
)

print("Extracted domains:")
print(df)

Task 4
Extracted domains:
               Email   Domain
0    user1@gmail.com    gmail
1    user2@yahoo.com    yahoo
2  user3@outlook.com  outlook


## 7. Export Functions

In [37]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': ['x', 'y', 'z'],
    'C': [1.1, 2.2, 3.3]
})

df.to_csv('output.csv', index=False)

df.to_excel(
    'output.xlsx',
    sheet_name='Results',
    index=False
)

df.to_json(
    'output.json',
    orient='records',
    indent=2
)

df.to_parquet(
    'output.parquet',
    index=False
)

df2 = pd.read_csv('output.csv')

print("CSV round-trip successful:")
print(df.equals(df2))

with pd.ExcelWriter('multi_sheet.xlsx') as writer:

    df.to_excel(
        writer,
        sheet_name='Data',
        index=False
    )

    df.describe().to_excel(
        writer,
        sheet_name='Stats'
    )

CSV round-trip successful:
True


## 7.2 Tasks — Export 

Task 1: Complete Titanic EDA: clean → feature engineer → groupby analysis → export clean CSV and an Excel with 2 sheets (Data + Stats). 

Task 2: Save three different subsets of Titanic (by pclass) to separate CSV files programmatically using a loop. 

In [38]:
print("Task 1")

df = sns.load_dataset('titanic')

df['age'] = df['age'].fillna(df['age'].median())

df['FamilySize'] = (
    df['sibsp'] + df['parch'] + 1
)

group_stats = df.groupby('pclass').agg(
    avg_age=('age', 'mean'),
    avg_fare=('fare', 'mean'),
    survival_rate=('survived', 'mean')
)

print("Group analysis:")
print(group_stats)

df.to_csv(
    'titanic_clean.csv',
    index=False
)

with pd.ExcelWriter('titanic_analysis.xlsx') as writer:

    df.to_excel(
        writer,
        sheet_name='Data',
        index=False
    )

    df.describe().to_excel(
        writer,
        sheet_name='Stats'
    )

Task 1
Group analysis:
          avg_age   avg_fare  survival_rate
pclass                                     
1       36.812130  84.154687       0.629630
2       29.765380  20.662183       0.472826
3       25.932627  13.675550       0.242363


In [39]:
print("Task 2")

df = sns.load_dataset('titanic')

for pclass in df['pclass'].unique():

    subset = df[df['pclass'] == pclass]

    subset.to_csv(
        f'titanic_class_{pclass}.csv',
        index=False
    )

print("CSV files created for all passenger classes.")

Task 2
CSV files created for all passenger classes.
